## Lab 11: Embeddings and Vision Transformer
*Suggested time: 35-40 minutes*


We show demonstrate the following models,

- A.  Image and Audio  Embeddings
- B. Vision Transformer (ViT)
- C. Conversion of ViT to ONNX
- D. Comparision of CNN (ResNet18) and ViT model


#### A. Image and Audio Embeddings

##### Step 0:

- Get embeddings from MobileNet_v3 model
- Upload image files for experimentation

In [0]:
!wget -O embedder.tflite -q https://storage.googleapis.com/mediapipe-models/image_embedder/mobilenet_v3_small/float32/1/mobilenet_v3_small.tflite

In [0]:
!wget https://edge-ai-doulos.s3.us-west-2.amazonaws.com/images.zip
!unzip -q images.zip

##### Step 1: Preparing Images
Now that you have retrieved the two images that will be compared, you can display them to confirm that they look correct. For this example you should see two separate, but similar, pictures of dogs and cats.

In [0]:
import cv2
import math
import matplotlib.pyplot as plt
import os
import path

BASE_DIR = os.getcwd()

IMAGE_FILENAMES = ['cat.png', 'macaw.png']


DESIRED_HEIGHT = 480
DESIRED_WIDTH = 480

def cv2_imshow(img):
  plt.figure(figsize=(8, 8))
  if len(img.shape) == 2:
    plt.imshow(img, cmap='gray')
  else:
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
  plt.axis('off')
  plt.show()
    
def resize_and_show(image):
  if image is None:
    print("Could not load image.")
    return

  h, w = image.shape[:2]
  if h < w:
    img = cv2.resize(image, (DESIRED_WIDTH, math.floor(h / (w / DESIRED_WIDTH))))
  else:
    img = cv2.resize(image, (math.floor(w / (h / DESIRED_HEIGHT)), DESIRED_HEIGHT))
  cv2_imshow(img)


# Preview the images.
for name in IMAGE_FILENAMES:
  full_path = os.path.join(BASE_DIR, 'images', name)
  print(full_path)
  image = cv2.imread(full_path)
  resize_and_show(image)

##### Step 2: Create and Compare Image Embeddings

Once everything looks good, you can start performing inference. You will start by creating the options that are necessary for associating your model with the Image Embedder, as well as some customizations.

Next you will create the Image Embedder, then format your two images for MediaPipe so that you can use cosine similarity to compare them.Finally, you will display the similarity value.

In [0]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os

# Create options for Image Embedder
base_options = python.BaseOptions(model_asset_path='embedder.tflite')
l2_normalize = True
quantize = True
options = vision.ImageEmbedderOptions(
    base_options=base_options, l2_normalize=l2_normalize, quantize=quantize)

# Use full image paths
first_path = os.path.join(BASE_DIR, 'images', IMAGE_FILENAMES[0])
second_path = os.path.join(BASE_DIR, 'images', IMAGE_FILENAMES[1])

# Create Image Embedder
with vision.ImageEmbedder.create_from_options(options) as embedder:
  first_image = mp.Image.create_from_file(first_path)
  second_image = mp.Image.create_from_file(second_path)

  first_embedding_result = embedder.embed(first_image)
  second_embedding_result = embedder.embed(second_image)

  similarity = vision.ImageEmbedder.cosine_similarity(
      first_embedding_result.embeddings[0],
      second_embedding_result.embeddings[0])
  print(similarity)

In [0]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import urllib

# Extract the embeddings as NumPy arrays for burger images
embedding1_np = np.array(first_embedding_result.embeddings[0].embedding)
embedding2_np = np.array(second_embedding_result.embeddings[0].embedding)

all_embeddings_list = [embedding1_np, embedding2_np]
labels_for_plot = ['Image 1 ', 'Image 2 ']
plot_markers = ['o', 'o']

combined_embeddings = np.vstack(all_embeddings_list)

print(f"Embedding 1 Dimensions : {embedding1_np.shape}")
print(f"Embedding 2 Dimensions: {embedding2_np.shape}")

# Initialize PCA to reduce to 2 components
pca_vision = PCA(n_components=2)

# Fit PCA to the combined embeddings and transform them
pca_result_vision = pca_vision.fit_transform(combined_embeddings)

# Create a scatter plot of the PCA-transformed embeddings
plt.figure(figsize=(10, 8))
for i in range(len(all_embeddings_list)):
    plt.scatter(pca_result_vision[i, 0], pca_result_vision[i, 1], label=labels_for_plot[i], s=100, marker=plot_markers[i])

plt.title('Vision Embeddings PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.legend()
plt.show()

print(f"PCA-transformed vision embedding shape: {pca_result_vision.shape}")
print(f"Explained variance ratio by principal components: {pca_vision.explained_variance_ratio_}")

##### Step 3:  Review of YAMNet acoustic model

- **Input format**
  - Accepts a **1-D float32 Tensor or NumPy array**.
  - The waveform can be **any length**.
  - Audio must be **mono, 16 kHz**, and scaled to the range **[-1.0, +1.0]**.

- **How the model processes audio**
  - The waveform is split into **0.96-second frames**.
  - Frames overlap with a **0.48-second hop**.
  - The model then processes all framed segments in a batch.

- **Returned outputs**
  - The model returns a **3-tuple**:
    - `scores`
    - `embeddings`
    - `log_mel_spectrogram`

- **`scores` output**
  - Shape: **(N, 521)**
  - `N` is the number of framed audio windows.
  - Contains **per-frame prediction scores** for the **521 supported AudioSet classes**.
  - Can be used to detect audio events by aggregating across frames, such as with **mean** or **max**.

- **`embeddings` output**
  - Shape: **(N, 1024)**
  - Contains **per-frame embedding vectors**.
  - These embeddings are the **average-pooled features** used before the final classifier layer.
  - Useful for:
    - building larger models
    - using YAMNet as a **feature extractor**
    - training shallow downstream classifiers

- **`log_mel_spectrogram` output**
  - Represents the **log mel spectrogram** of the full waveform.
  - Shape: **(num_spectrogram_frames, 64)**
  - Computed using:
    - **0.025-second analysis windows**
    - **0.01-second hop**
    - **64 mel bins**
  - Mainly useful for **visualization** and **debugging**.

- **Class mapping**
  - Each column in `scores` corresponds to an **AudioSet class**.
  - Column indices **0–520** map to class names using the **YAMNet Class Map**.
  - The class map is available:
    - as a **CSV file** in the GitHub repository


##### Step 4:  Install TensorFlow Hub 

In [0]:
!pip install -q tensorflow_hub

##### Step 5:  Assert shapes of scores, embeddings and spectrograms of YAMNet model

In [0]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import csv
import io

# Load the model.
model = hub.load('https://www.kaggle.com/models/google/yamnet/TensorFlow2/yamnet/1')

# Input: 3 seconds of silence as mono 16 kHz waveform samples.
waveform = np.zeros(3 * 16000, dtype=np.float32)

# Run the model, check the output.
scores, embeddings, log_mel_spectrogram = model(waveform)
scores.shape.assert_is_compatible_with([None, 521])
embeddings.shape.assert_is_compatible_with([None, 1024])
log_mel_spectrogram.shape.assert_is_compatible_with([None, 64])

# Print the embeddings vector as a NumPy array
print(embeddings.numpy().shape)
print (scores.numpy().shape)
print ('Spectrogram shape', log_mel_spectrogram.numpy().shape)
print ('Spectrogram values', log_mel_spectrogram.numpy())

##### Step 6:  Visualize YAMNet Embeddings with PCA

- PCA is used to reduce YAMNet’s 1024-dimensional embeddings to 2 or 3 dimensions.
- This dimensionality reduction makes the embeddings easier to visualize on 2D or 3D plots.
- Visualizing the embeddings can help reveal clusters or patterns in the audio representation space.

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# Ensure embeddings are a NumPy array
embeddings_np = embeddings.numpy()

# Initialize PCA to reduce to 2 components for 2D plotting
pca = PCA(n_components=2)

# Fit PCA to the embeddings and transform them
# Each row in embeddings_np corresponds to an audio frame's embedding
pca_result = pca.fit_transform(embeddings_np)

# Create a scatter plot of the PCA-transformed embeddings
plt.figure(figsize=(10, 8))
plt.scatter(pca_result[:, 0], pca_result[:, 1], alpha=0.7)
plt.title('YAMNet Embeddings PCA (Silence Input)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
# Removed xlim and ylim to zoom out
plt.show()

print(f"Original embedding shape: {embeddings_np.shape}")
print(f"PCA-transformed embedding shape: {pca_result.shape}")
print(f"Explained variance ratio by principal components: {pca.explained_variance_ratio_}")

##### Step 7: Generate YAMNet Embeddings for Multiple Sound Waves and Visualize with PCA

This section demonstrates how to generate embeddings for different types of sound waves (silence, sine wave, random noise) using the YAMNet model, and then visualize these high-dimensional embeddings in a 2D space using Principal Component Analysis (PCA). This helps in understanding how YAMNet distinguishes between different audio patterns.

In [0]:
import numpy as np
import tensorflow as tf

# Generate different waveforms
sample_rate = 16000 # YAMNet expects 16 kHz audio
duration = 3 # seconds

# 1. Silence
waveform_silence = np.zeros(int(duration * sample_rate), dtype=np.float32)

# 2. Sine wave (e.g., 440 Hz tone)
frequency_sine = 440 # Hz
t = np.linspace(0, duration, int(duration * sample_rate), endpoint=False)
waveform_sine = 0.5 * np.sin(2 * np.pi * frequency_sine * t).astype(np.float32)

# 3. Random noise
waveform_noise = np.random.uniform(-0.5, 0.5, int(duration * sample_rate)).astype(np.float32)

# 4. Square wave (e.g., 220 Hz)
frequency_square = 220 # Hz
waveform_square = 0.5 * np.sign(np.sin(2 * np.pi * frequency_square * t)).astype(np.float32)

# 5. Sawtooth wave (e.g., 100 Hz)
frequency_sawtooth = 100 # Hz
waveform_sawtooth = 0.5 * (2 * (t * frequency_sawtooth - np.floor(t * frequency_sawtooth + 0.5))).astype(np.float32)

waveforms = {
    "Silence": waveform_silence,
    "Sine Wave (440 Hz)": waveform_sine,
    "Random Noise": waveform_noise,
    "Square Wave (220 Hz)": waveform_square,
    "Sawtooth Wave (100 Hz)": waveform_sawtooth
}

print("Generated waveforms:")
for name, wf in waveforms.items():
    print(f"- {name}: {wf.shape} samples")

In [0]:
all_embeddings = []
embedding_labels = []

# Ensure the YAMNet model is loaded
# If model is not defined, load it here
if 'model' not in locals():
    import tensorflow_hub as hub
    model = hub.load('https://www.kaggle.com/models/google/yamnet/TensorFlow2/yamnet/1')


for label, waveform in waveforms.items():
    # YAMNet can handle arbitrary length, but internally frames it.
    # It returns one embedding per frame, so we'll average them.
    scores, embeddings, log_mel_spectrogram = model(waveform)

    # Take the mean of embeddings if YAMNet returns multiple frames
    if embeddings.shape[0] > 1:
        averaged_embedding = np.mean(embeddings.numpy(), axis=0)
    else:
        averaged_embedding = embeddings.numpy()[0]

    all_embeddings.append(averaged_embedding)
    embedding_labels.append(label)

embeddings_array = np.array(all_embeddings)

print(f"Collected {len(all_embeddings)} embeddings, each of shape {embeddings_array.shape[1]}")

In [0]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Initialize PCA to reduce to 2 components
pca_multi = PCA(n_components=2)

# Fit PCA to the collected embeddings and transform them
pca_result_multi = pca_multi.fit_transform(embeddings_array)

# Create a scatter plot of the PCA-transformed embeddings
plt.figure(figsize=(10, 8))
for i, (x, y) in enumerate(pca_result_multi):
    plt.scatter(x, y, label=embedding_labels[i], s=100) # s is marker size
    plt.annotate(embedding_labels[i], (x + 0.05, y + 0.05)) # Add label next to point

plt.title('YAMNet Embeddings PCA for Different Sound Waves')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.legend()
plt.show()

print(f"Original embeddings shape: {embeddings_array.shape}")
print(f"PCA-transformed embeddings shape: {pca_result_multi.shape}")
print(f"Explained variance ratio by principal components: {pca_multi.explained_variance_ratio_}")

##### Step 8:  Execute YAMNet model using LiteRT runtime

In [0]:
!pip install -q ai-edge-litert

In [0]:
!wget -O sound-classifier.tflite -q https://storage.googleapis.com/mediapipe-models/audio_classifier/yamnet/float32/1/yamnet.tflite

In [0]:
from ai_edge_litert.interpreter import Interpreter

import tensorflow as tf
import numpy as np
import zipfile

model_path = ('sound-classifier.tflite')
interpreter = Interpreter(model_path)

input_details = interpreter.get_input_details()
waveform_input_index = input_details[0]['index']
output_details = interpreter.get_output_details()
scores_output_index = output_details[0]['index']

# Input: 0.975 seconds of silence as mono 16 kHz waveform samples.
waveform = np.zeros(int(round(0.975 * 16000)), dtype=np.float32)
#waveform = np.zeros(int(round(3.0 * 16000)), dtype=np.float32)
print(waveform.shape)  # Should print (15600,)

interpreter.resize_tensor_input(waveform_input_index, [waveform.size], strict=True)
interpreter.allocate_tensors()
interpreter.set_tensor(waveform_input_index, waveform)
interpreter.invoke()
scores = interpreter.get_tensor(scores_output_index)
print(scores.shape)  # Should print (1, 521)

top_class_index = scores.argmax()
labels_file = zipfile.ZipFile(model_path).open('yamnet_label_list.txt')
labels = [l.decode('utf-8').strip() for l in labels_file.readlines()]
print(len(labels))  # Should print 521
print(labels[top_class_index])  # Should print 'Silence'.

#### B.  Vision Transformer (ViT) model

##### Step 0 :  Install needed Python packages
- **Transformers** (Maintained by Hugging Face. It provides a unified API for downloading, training, and deploying thousands of pretrained models across multiple modalities, including text, vision, and audio.)
- **torch** (core Python package for PyTorch)

In [3]:
!pip install -q transformers torch


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


##### Step 1: Inferencing using  Vision Transformer (ViT) 

ViT used:  vit-base-patch16-224
- vit: Vision Transformer architecture.
- base: Base-sized model parameters (~86 million total parameters).
- patch16: Images are divided into non-overlapping patches of (16x16) pixels.
- 224: Standard expected input resolution of  (224x224) pixels.


In [12]:
from transformers import pipeline
from PIL import Image
import requests

# Load the ViT pipeline
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

# Load an image (e.g., a cat)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Predict
results = classifier(image)
for result in results:
    print(f"{result['label']}: {round(result['score'], 4)}")

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Device set to use cpu


Egyptian cat: 0.9374
tabby, tabby cat: 0.0384
tiger cat: 0.0144
lynx, catamount: 0.0033
Siamese cat, Siamese: 0.0007


##### **Step 2:**
Install the optimum-cli tool and convert ViT model to ONNX format

In [0]:
!pip install -q "optimum[onnxruntime]"

In [0]:
!optimum-cli export onnx --model google/vit-base-patch16-224 vit_onnx/

In [0]:
from optimum.onnxruntime import ORTModelForImageClassification
from transformers import AutoFeatureExtractor, pipeline
from PIL import Image
import requests

# 1. Load the converted ONNX model instead of the PyTorch one
model_path = "vit_onnx/"
model = ORTModelForImageClassification.from_pretrained(model_path)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_path)

# 2. Use the same pipeline API
classifier = pipeline("image-classification", model=model, feature_extractor=feature_extractor)

# 3. Predict (the rest of your code remains the same)
#url = "http://images.cocodataset.org/val2017/000000000632.jpg"
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
image.save("cat.jpg")

results = classifier(image)
for result in results:
    print(f"{result['label']}: {round(result['score'], 4)}")

##### **Step 3:**
- Create a obfuscated image from moving around blocks of pixels
- Check the efficacy of ViT model and a ResNet model to correctly classify the image
- Start with installing LiteRT runtime and converting CNN and ViT models to Lite

```text
      [ INPUT IMAGE ]
             |
             v
    +-----------------+
    |  Pre-processing | (Resize to 224x224, BGR -> RGB)
    +-----------------+
             |
      _______|_______
     |               |
     v               v
 [ PATH A ]      [ PATH B ]
  Original       Obfuscated
   Image           Image
     |               |
     |        +--------------+
     |        | Patch & Mix  | (Split into 56px patches,
     |        |   Shuffle    |  Reconstruct scrambled)
     |        +--------------+
     |               |
     v               v
+-----------------------------+
|   TFLite Inference Engine   |
+-----------------------------+
|                             |
|  [ RESNET-50 (CNN) ]        | --> Result: Likely Fails on Path B
|    (Relies on Local         |              (Shapes broken)
|     Connectivity)           |
|                             |
|  [ ViT-BASE (Transformer) ] | --> Result: Likely Succeeds on Path B
|    (Relies on Global        |              (Attends to patches
|     Attention)              |               anywhere)
+-----------------------------+
             |
             v
      [ PRINT RESULTS ]
   (Compare Argmax Classes)
   ```

In [0]:
!pip install -q ai-edge-litert


In [0]:
!pip uninstall -q keras-hub


In [0]:
!curl -o imagenet_labels.json https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json

In [0]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"  # Or "tensorflow" or "torch"!

In [0]:
import numpy as np
import cv2
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter
import keras_hub

# 1. Load the model from Keras Hub
model = keras_hub.models.ImageClassifier.from_preset(
    "resnet_18_imagenet",
    activation="softmax"
)

# 2. Initialize the converter
# from_keras_model is the recommended entry point for TF2/TF3 backends
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 3. Optional: Post-Training Quantization (reduces size by ~4x)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 4. Convert and Save
tflite_model = converter.convert()
with open("resnet18.tflite", "wb") as f:
    f.write(tflite_model)

# 1. Load ViT model
# Note: ViT usually requires a specific 224x224 input
vit_model = keras_hub.models.ImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet",
    activation="softmax"
)

converter = tf.lite.TFLiteConverter.from_keras_model(vit_model)

# 2. Enable 'Select TF Ops' for Transformer compatibility
# This allows TFLite to 'fall back' to regular TensorFlow for complex attention ops
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, # Enable standard TFLite ops
    tf.lite.OpsSet.SELECT_TF_OPS    # Enable TensorFlow ops fallback
]

# 3. Convert and Save
tflite_vit_model = converter.convert()
with open("vit_base.tflite", "wb") as f:
    f.write(tflite_vit_model)


In [0]:
import numpy as np
import cv2
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter
import keras_hub
import matplotlib.pyplot as plt
import json

def obfuscate_image(image, patch_size=32):
    """Shuffles patches of the image to obfuscate it."""
    h, w, c = image.shape
    patches = []
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            patches.append(image[i:i+patch_size, j:j+patch_size])

    np.random.shuffle(patches)

    # Reconstruct the image
    obfuscated = np.zeros_like(image)
    idx = 0
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            obfuscated[i:i+patch_size, j:j+patch_size] = patches[idx]
            idx += 1
    return obfuscated


def run_tflite_inference(model_path, image):
    """Standard TFLite inference loop."""

    interpreter = Interpreter(model_path=model_path)
    # interpreter.allocate_tensors() # Tensors will be allocated after resize

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # 1. Determine the target height and width for the input image
    # If the model's reported shape is 1x1 (often a placeholder), default to 224x224
    input_shape = input_details[0]['shape']
    model_h, model_w = input_shape[1], input_shape[2]

    target_h, target_w = 224, 224 # Default to 224x224 as it's common for these models
    if model_h > 1 and model_w > 1: # If model reports a valid size (e.g., 224x224), use it
        target_h, target_w = model_h, model_w

    # 2. Resize the interpreter's input tensor based on the determined target dimensions
    # This must be called BEFORE allocate_tensors()
    interpreter.resize_tensor_input(input_details[0]['index'], [1, target_h, target_w, 3])
    interpreter.allocate_tensors() # Allocate tensors *after* resizing

    # 3. Perform the image resize
    resized = cv2.resize(image, (target_w, target_h))

    # 4. Prepare for the model
    input_data = np.expand_dims(resized, axis=0).astype(np.float32)
    input_data = input_data / 255.0

    interpreter.set_tensor(input_details[0]['index'], input_data)
    #print(f"Input Min/Max: {input_data.min()}, {input_data.max()}")
    #print(f"Input Shape: {input_data.shape}")
    interpreter.invoke()

    return interpreter.get_tensor(output_details[0]['index'])[0]

# --- Main Logic ---
image_path = "images/clock.jpg"
image = cv2.imread(image_path)

# Load ImageNet labels
with open('imagenet_labels.json', 'r') as f:
    imagenet_labels = json.load(f)

# Check if image was loaded correctly
if image is None:
    print(f"Error: Could not load image from {image_path}")
else:
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Resize image_rgb to be a multiple of patch_size (56) for obfuscation
    target_dim = 224 # 224 is 4 * 56
    image_rgb_resized = cv2.resize(image_rgb, (target_dim, target_dim))

    # 1. Obfuscate
    patch_size = 56
    obfuscated_rgb = obfuscate_image(image_rgb_resized, patch_size=patch_size)

    # Display side-by-side
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(image_rgb_resized)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(obfuscated_rgb)
    plt.title(f"Obfuscated - {patch_size}px Patches")
    plt.axis("off")

    plt.show()

    # 2. Classify with ResNet (CNN)
    # Pass the resized image for consistent input dimensions
    resnet_out = run_tflite_inference("resnet18.tflite", image_rgb_resized )
    resnet_original_class_id = np.argmax(resnet_out)
    print(f"ResNet Original Top Class: {resnet_original_class_id} ({imagenet_labels[str(resnet_original_class_id)][1]})")

    resnet_out_ob = run_tflite_inference("resnet18.tflite", obfuscated_rgb)
    resnet_obfuscated_class_id = np.argmax(resnet_out_ob)
    print(f"ResNet Obfuscated Top Class: {resnet_obfuscated_class_id} ({imagenet_labels[str(resnet_obfuscated_class_id)][1]})")


    # 3. Classify with ViT (Transformer)
    vit_out = run_tflite_inference("vit_base.tflite", image_rgb_resized)
    vit_original_class_id = np.argmax(vit_out)
    print(f"ViT Original Top Class: {vit_original_class_id} ({imagenet_labels[str(vit_original_class_id)][1]})")

    vit_out_g = run_tflite_inference("vit_base.tflite", obfuscated_rgb)
    vit_obfuscated_class_id = np.argmax(vit_out_g)
    print(f"ViT Obfuscated Top Class: {vit_obfuscated_class_id} ({imagenet_labels[str(vit_obfuscated_class_id)][1]})")

In [0]:
!pip install -q transformers torch

In [0]:
!wget https://edge-ai-doulos.s3.us-west-2.amazonaws.com/images.zip
!unzip -q images.zip

In [0]:
from transformers import AutoProcessor, CLIPModel
import torch
from PIL import Image
import requests

# Explicitly set the device to CPU
device = "cpu"
print(f"Using device: {device}")

# Load the model and processor
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = AutoProcessor.from_pretrained(model_name)

# Example: Perform zero-shot image classification

image_location = 'images/cat.png'
image = Image.open(image_location).convert("RGB")

candidate_labels = [
    "a photo of a cat", "a photo of a dog", "a photo of an elephant"
]

# Process inputs
inputs = processor(
    text=candidate_labels,
    images=image,
    return_tensors="pt",
    padding=True,
)

# Move tensor inputs to the CPU only
inputs = {
    k: v.to(device) if torch.is_tensor(v) else v
    for k, v in inputs.items()
}

# Run inference
with torch.no_grad():
    outputs = model(**inputs)

# Get the results
logits_per_image = outputs.logits_per_image
probs = logits_per_image.softmax(dim=1)
print(f"Probabilities: {probs}")

In [0]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image_location = 'images/two_dogs.png'
#url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(image_location)

inputs = processor(text=["picture of elephant", "a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities
print(f"Probabilities: {probs}")

In [0]:
# DINO (Self-Supervised ViT)
import torch
from PIL import Image
import requests

# Load DINO ViT-Small (pretrained without labels)
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
model.eval()

# Use this to visualize how the model automatically segments objects
# from the background without ever being told what a "cat" or "car" is.

In [0]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import requests
import matplotlib.pyplot as plt
import numpy as np

# Load DINO ViT-Small (8x8 patches for higher resolution detail)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8').to(device)
model.eval()

# Preprocessing: DINO expects 224x224 or multiples of the patch size (8)
transform = T.Compose([
    T.Resize(480), # High res for better visualization
    T.CenterCrop(480),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Load an image (Change the URL to your own or use a local path)
url = "https://dl.fbaipublicfiles.com/dino/img.png"
image_location = 'images/macaw_2.png'
img_raw = Image.open(image_location).convert('RGB')
img_tensor = transform(img_raw).unsqueeze(0).to(device)

- The different DINO attention heads visualize distinct semantic regions or features learned by the model without explicit supervision.
- In self-supervised Vision Transformers, each attention head can specialize in a different way of relating image patches to the `[CLS]` token.
- Common emergent behaviors include:
  - **Object segmentation:** some heads strongly attend to the foreground object, separating it from the background.
  - **Background attention:** other heads focus on contextual background regions.
  - **Feature detection:** some heads respond to textures, shapes, or object parts regardless of exact position.
  - **Diversity of focus:** each head produces a unique attention pattern, giving the model richer visual representation.
- In the plotted subplots, each head shows one attention map from DINO.
- The varying intensity patterns across heads suggest unsupervised object discovery or segmentation-like behavior.


In [0]:
# 1. Forward pass to get attention
# We use 'get_last_selfattention' which is a built-in method for the DINO hub model
with torch.no_grad():
    attentions = model.get_last_selfattention(img_tensor)

# 2. Process the attention map
# Structure: [Batch, Heads, Num_Patches+1, Num_Patches+1]
nh = attentions.shape[1] # Number of attention heads (usually 6 for ViT-S)
w_featmap = img_tensor.shape[-2] // 8
h_featmap = img_tensor.shape[-1] // 8

# We keep only the attention of the [CLS] token to the other patches
# Index 0 is the CLS token; we take 0, :, 0, 1:
attentions = attentions[0, :, 0, 1:].reshape(nh, w_featmap, h_featmap)

# 3. Visualize the Different "Heads"
fig, axs = plt.subplots(1, nh + 1, figsize=(20, 5))
axs[0].imshow(img_raw.resize((480, 480)))
axs[0].set_title("Original")
axs[0].axis('off')

for i in range(nh):
    axs[i+1].imshow(attentions[i].cpu().numpy(), cmap='magma')
    axs[i+1].set_title(f"Head {i}")
    axs[i+1].axis('off')

plt.show()

The images showing **'Head 0' to 'Head 5'** are visualizations of the **attention heads from the last Transformer block** of the DINO model.

- The notebook uses `model.get_last_selfattention(img_tensor)` to extract the attention maps.
- This method returns the **self-attention weights from the final layer** of the Vision Transformer encoder.
- The last-layer attention maps are often the most **semantically meaningful**, so they are useful for interpreting what the model focuses on.
- Each head (`Head 0` to `Head 5`) represents a different attention pattern learned by the model.

- The number of attention heads (nh) in a DINO model can be determined from the shape of the extracted attention tensor.
- After a forward pass, attentions has shape [Batch, Heads, Num_Patches+1, Num_Patches+1].
- The number of heads is obtained with nh = attentions.shape[1].
- In the example using dino_vits8, the model has attention heads.


In [0]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from PIL import Image
import requests
import torchvision.transforms as T

# 1. Load Model (DINOv2 is highly recommended for this)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
model.eval()

# 2. Load and Preprocess Image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = Image.open(requests.get(url, stream=True).raw).convert('RGB')
w, h = img.size
# Resize to a multiple of patch size (14)
img_t = T.Compose([
    T.Resize((448, 448)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])(img).unsqueeze(0).to(device)

# 3. Extract Patch Features
with torch.no_grad():
    features_dict = model.forward_features(img_t)
    features = features_dict['x_norm_patchtokens'] # Shape: [1, 1024, 384]

# 4. Perform PCA (Reduce 384-D -> 3-D)
# Flatten to [Num_Patches, Embedding_Dim]
features = features.squeeze(0).cpu().numpy()
pca = PCA(n_components=3)
pca_features = pca.fit_transform(features)

# 5. Normalize and Reshape to RGB Image
# Scale features to 0-1 range for RGB visualization
pca_features = (pca_features - pca_features.min()) / (pca_features.max() - pca_features.min())

# Reshape back to grid (448/14 = 32 patches)
grid_size = 448 // 14
pca_img = pca_features.reshape(grid_size, grid_size, 3)

# 6. Plot Results
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img.resize((448, 448)))
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(pca_img)
plt.title("DINO PCA Segmentation")
plt.axis('off')
plt.show()

In [0]:
!pip install -q https://github.com/roboflow/rf-detr/archive/refs/tags/1.5.0.rc1.zip

In [0]:
import os

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Scale data-loader workers with available CPUs so the GPU is kept fed.
num_workers = max(os.cpu_count(), 2)
print(f"Data loader workers: {num_workers}")

In [0]:
!pip uninstall transformers -y
!pip install transformers==5.6.2

In [0]:
!pip install -q supervision

In [0]:
!pip install -q rfdetr

In [0]:
import requests
import supervision as sv
from PIL import Image
from rfdetr import RFDETRNano
from rfdetr.util.coco_classes import COCO_CLASSES

model = RFDETRNano()

image = "clock.jpg"
image = Image.open(image_path).convert("RGB")

detections = model.predict(image, threshold=0.5)

labels = [f"{COCO_CLASSES[class_id]}" for class_id in detections.class_id]

annotated_image = sv.BoxAnnotator().annotate(image, detections)
annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)

sv.plot_image(annotated_image)

In [0]:
import cv2
import requests
import supervision as sv
from PIL import Image
from io import BytesIO
from rfdetr import RFDETRBase

# 1. Load the pre-trained Base model (trained on COCO)
model = RFDETRBase()

# 2. Load an image from a URL
image_path = "pizza.png"
image = Image.open(image_path).convert("RGB") # Ensure image is in RGB format

# 3. Predict
# Returns a supervision Detections object
detections = model.predict(image, threshold=0.5)

# 4. Annotate and Plot
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

sv.plot_image(annotated_image)

In [0]:
import torch
from PIL import Image
import requests
from transformers import AutoImageProcessor, ViTModel

# 1. Load the model and processor
model_name = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(model_name)
model = ViTModel.from_pretrained(model_name)

# 2. Load and preprocess an image
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt")

# 3. Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# 4. Extract the [CLS] token
# last_hidden_state shape: [batch_size, sequence_length, hidden_size]
# sequence_length = (number_of_patches + 1) -> 196 + 1 = 197
last_hidden_state = outputs.last_hidden_state
cls_token = last_hidden_state[:, 0, :]

print(f"Full Hidden State Shape: {last_hidden_state.shape}")
print(f"Extracted [CLS] Token Shape: {cls_token.shape}")

In [0]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
import numpy as np

# Load DINOv2 (Small version is fastest for Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
model = AutoModel.from_pretrained("facebook/dinov2-small").to(device)

@torch.no_grad()
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    outputs = model(**inputs)
    # The [CLS] token is the first token in the last hidden state
    embedding = outputs.last_hidden_state[:, 0, :]
    return embedding.cpu().numpy().flatten()

In [0]:
import os
os.environ["KERAS_BACKEND"] = "jax" # JAX is recommended for Transformers in 2026

import keras_hub
import keras
import numpy as np
from PIL import Image

# Print available presets to debug the error
print("Available ObjectDetector presets:", keras_hub.models.ObjectDetector.presets.keys())

# 1. Load a SOTA Transformer-based Object Detector
# Preset "dfine_s_obj2coco" is a Small D-FINE model pretrained on Objects365 & COCO
detector = keras_hub.models.ObjectDetector.from_preset(
    "retinanet_resnet50_fpn_coco", # Changed to a valid preset
    bounding_box_format="xywh"
)

# 2. Load and Preprocess an Image
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c4/Savannah_Cat_portrait.jpg/640px-Savannah_Cat_portrait.jpg"
image_path = keras.utils.get_file(origin=url)
image = keras.utils.load_img(image_path, target_size=(640, 640))
image_array = keras.utils.img_to_array(image)

# 3. Run Inference
# The output contains 'boxes', 'confidence', and 'classes'
outputs = detector.predict(np.expand_dims(image_array, axis=0))

# 4. Access the Detections
boxes = outputs["boxes"][0]
classes = outputs["classes"][0]
confidences = outputs["confidence"][0]

print(f"Detected {len(boxes[confidences > 0.5])} objects with >50% confidence.")

In [0]:
!pip install -U keras-hub
!pip install -U keras

In [0]:
!pip install keras-cv
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import requests
import matplotlib.pyplot as plt
import numpy as np

import os
os.environ["KERAS_BACKEND"] = "jax" # JAX is recommended for Transformers in 2026

import keras_hub
import keras
import keras_cv # Added import for keras_cv
import numpy as np
from PIL import Image

# Pretrained DINOV3 model.
input_data = {
    "pixel_values": np.ones(shape=(1, 224, 224, 3), dtype="float32"), # Changed to 224x224
}
model = keras_hub.models.DINOV3Backbone.from_preset(
    "dinov3_vit_small_lvd1689m"
)
model(input_data)

# Pretrained DINOV3 model with custom image shape.
input_data = {
    "pixel_values": np.ones(shape=(1, 224, 224, 3), dtype="float32"), # Changed key from 'images' to 'pixel_values'
}
model = keras_hub.models.DINOV3Backbone.from_preset(
    "dinov3_vit_small_lvd1689m", image_shape=(224, 224, 3)
)
model(input_data)

# Randomly initialized DINOV3 model with custom config.
model = keras_hub.models.DINOV3Backbone(
    patch_size=14,
    num_layers=2,
    hidden_dim=32,
    num_heads=2,
    intermediate_dim=128,
    image_shape=(224, 224, 3),
)
model(input_data)

# Accessing feature pyramid outputs.
backbone = keras_hub.models.DINOV3Backbone.from_preset(
    "dinov3_vit_small_lvd1689m", image_shape=(224, 224, 3)
)
model = keras.Model(
    inputs=backbone.inputs,
    outputs=backbone.pyramid_outputs,
)
features = model(input_data)

print (features)


In [0]:
!pip install ai-edge-litert

In [0]:
!pip install -U -q keras-hub

```text
       [ Input Image ]  (e.g., 640x640)
             |
             v
[ Initial Convolution (Conv2d, 3x3) ]  (Basic feature extraction)
             |
             v
[ HGNetv2 Backbone Block ]  (Repeated K times, e.g., using lightweight HGBottleneck)
             |
             v
[ Final Convolution & Global Pooling (HGNetv2 Output) ]  (High-level features C3, C4, C5)
             |
             v
[ Feature Pyramid Fusion (FPN/PANet-like) ]  (Multi-scale feature combination)
             |
             v
[ Hybrid Encoder (GELAN Block) ]  (Deep feature interaction)
|           / \
|  [ Aggregation Layer (HGBottleneck) ]  (Local & global feature integration)
|          \   /
|           v
| [ Transformer Decoder ]  (Iterative query refinement)
| |           / \
| |  [ Learnable Object Queries ]  (Input queries)
| |           |
| |  [ Multi-Head Self-Attention (MSA) ]  (Query interaction)
| |           |
| |  [ Cross-Modality Fusion (Image-Text/Feature) ]  (Context alignment)
| |           |
| |  [ Feed-Forward Network (FFN) ]  (GELU activation)
| |           |
| |  [ Residual Connection ]
| |           |
|  -----------|
|             |
|-------------
              |
              v
 [ Detection Head (Classification & Bbox) ]
 ```

https://www.kaggle.com/models/keras/d-fine/keras/dfine_nano_coco/1

In [0]:
import keras
import keras_hub
import numpy as np
from keras_hub.models import DFineBackbone
from keras_hub.models import DFineObjectDetector
from keras_hub.models import HGNetV2Backbone
import cv2 # Added for cv2.imread, cvtColor, resize

object_detector = DFineObjectDetector.from_preset(
    "dfine_nano_coco"
)

# Create a random image.
#image = np.random.uniform(size=(1, 256, 256, 3)).astype("float32")

image_path = "pizza.png"
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (256, 256)) # Standard ViT size

# Add a batch dimension to the image
img_batch = np.expand_dims(img, axis=0)

# Make predictions.
predictions = object_detector.predict(img_batch)

# The output is a dictionary containing boxes, labels, confidence scores,
# and the number of detections.
print(predictions["boxes"].shape)
print(predictions["labels"].shape)
print(predictions["confidence"].shape)
print(predictions["num_detections"])

In [0]:
num_detections = predictions["num_detections"][0]
detected_labels = predictions["labels"][0][:num_detections]
detected_confidences = predictions["confidence"][0][:num_detections]

print("Detected Objects and Confidences:")
for i in range(num_detections):
    # Assuming COCO_CLASSES is available from a previous cell, as in other examples in the notebook
    # If COCO_CLASSES is not defined, this will raise an error. For now, we assume it exists.
    label_name = f"Class {detected_labels[i]}" # Fallback if COCO_CLASSES not available
    try:
        from rfdetr.util.coco_classes import COCO_CLASSES
        label_name = COCO_CLASSES[int(detected_labels[i])]
    except (ImportError, KeyError):
        pass # Keep fallback if import fails or class ID is not in COCO_CLASSES

    print(f"- {label_name}: {detected_confidences[i]:.4f}")